# Eksplorasi Data (Tabel Data)

**Tujuan:** Menampilkan isi data dalam bentuk **tabel** untuk eksplorasi —
mulai dari board mentah hasil parsing (sebelum feature engineering) sampai
dataset final `data/processed/` yang dipakai model.

**Input:** `data/raw/` (LIN), `data/raw_pbn/` (PBN), `data/processed/*.csv`
(output `01_data_extraction.ipynb` — jalankan notebook 1 dulu).

**Output:** tabel di notebook + `outputs/results/nb01b_data_dictionary.csv`.

Notebook ini **hanya membaca data** — tidak mengubah dataset, split, maupun model.

| Bagian | Isi |
|---|---|
| 1 | Tabel board mentah (tangan, lelang, kontrak, hasil) |
| 2 | Ukuran dataset & split |
| 3 | `head` / `tail` / `sample` per kelompok kolom |
| 4 | Kamus data (dtype, missing, unique, min/max) |
| 5 | Statistik deskriptif (`describe`) |
| 6 | Kualitas data: missing value & duplikat |
| 7 | Tabel frekuensi kolom kategorikal |
| 8 | Distribusi target `matches_par_contract` |
| 9 | Perbandingan fitur per kelas target |
| 10 | Contoh: satu papan fisik di open vs closed room |

---
## 0. Setup

In [1]:
import sys
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

warnings.filterwarnings('ignore')

ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT))

PROCESSED_DIR = ROOT / 'data' / 'processed'
OUT = ROOT / 'outputs' / 'results'
OUT.mkdir(parents=True, exist_ok=True)

pd.set_option('display.max_columns', 60)
pd.set_option('display.max_colwidth', 60)
pd.set_option('display.width', 200)
print('Setup selesai.')

Setup selesai.


In [2]:
# Load dataset final (hasil notebook 1)
read_kw = dict(keep_default_na=True, low_memory=False)
df       = pd.read_csv(PROCESSED_DIR / 'full.csv', **read_kw)
df_train = pd.read_csv(PROCESSED_DIR / 'train.csv', **read_kw)
df_val   = pd.read_csv(PROCESSED_DIR / 'val.csv', **read_kw)
df_test  = pd.read_csv(PROCESSED_DIR / 'test.csv', **read_kw)
feature_cols = json.loads((PROCESSED_DIR / 'feature_columns.json').read_text())

meta_cols    = [c for c in df.columns if c.startswith('_')]
target_cols  = [c for c in ['matches_par_contract', 'label', 'target_base', 'target', 'target_category']
                if c in df.columns]
leakage_cols = [c for c in df.columns if c.startswith('dd_par_')]

# Kolom bantu (hanya untuk eksplorasi, tidak disimpan)
source_type = pd.Series(
    np.where(df['_source_file'].astype(str).str.lower().str.endswith('.lin'), 'LIN (BBO)', 'PBN'),
    index=df.index, name='sumber',
)
print(f'full.csv: {df.shape[0]:,} baris x {df.shape[1]} kolom | fitur model: {len(feature_cols)}')

full.csv: 50,227 baris x 195 kolom | fitur model: 174


---
## 1. Tabel Board Mentah (Sebelum Feature Engineering)

Satu baris = satu board hasil parsing. Tangan ditulis per suit
(`♠ | ♥ | ♦ | ♣`), lelang ditulis urut dari dealer. Contoh diambil dari
beberapa file LIN pertama dan beberapa file PBN pertama saja (parsing semua
file dilakukan di notebook 1).

In [3]:
from src.parser import LINParser, PBNParser

lin_sample_files = sorted((ROOT / 'data' / 'raw').glob('*.lin'))[:3]
pbn_sample_files = sorted({p for p in (ROOT / 'data' / 'raw_pbn').iterdir()
                           if p.suffix.lower() == '.pbn'})[:2]

raw_boards = []
for f in lin_sample_files:
    raw_boards += LINParser().parse_file(f)
for f in pbn_sample_files:
    raw_boards += PBNParser().parse_file(f)

def hand_str(hand):
    if hand is None:
        return ''
    d = hand.as_dict()
    return ' | '.join(''.join(d.get(s, [])) or '-' for s in ('S', 'H', 'D', 'C'))

def contract_str(b):
    if not b.contract_level:
        return 'PASS'
    return f"{b.contract_level}{b.contract_strain}{'xx' if b.redoubled else 'x' if b.doubled else ''}"

raw_df = pd.DataFrame([{
    'file'     : Path(b.source_file).name,
    'room'     : b.room,
    'board'    : b.board_number,
    'dealer'   : b.dealer,
    'vul'      : b.vulnerability,
    'North'    : hand_str(b.hands.get('N')),
    'East'     : hand_str(b.hands.get('E')),
    'South'    : hand_str(b.hands.get('S')),
    'West'     : hand_str(b.hands.get('W')),
    'lelang'   : ' '.join(b.auction),
    'kontrak'  : contract_str(b),
    'declarer' : b.declarer,
    'hasil'    : b.result,
    'trik'     : b.tricks_made,
} for b in raw_boards])

print(f'{len(raw_boards)} board dari {len(lin_sample_files)} file LIN + {len(pbn_sample_files)} file PBN')
display(raw_df.head(15))

130 board dari 3 file LIN + 2 file PBN


,file,room,board,dealer,vul,North,East,South,West,lelang,kontrak,declarer,hasil,trik
0,85168.lin,open,13,N,Both,KT87 | T76 | 873 | 743,Q642 | J532 | 2 | AQ62,J9 | AQ9 | QJT96 | J95,A53 | K84 | AK54 | KT8,p p 1D 1N p 2C p 2D p 2N p 3N p p p,3N,W,-1.0,8.0
1,85168.lin,closed,13,N,Both,KT87 | T76 | 873 | 743,Q642 | J532 | 2 | AQ62,J9 | AQ9 | QJT96 | J95,A53 | K84 | AK54 | KT8,p p 1D 1N p 2C p 2D p 3N p p p,3N,W,NaN,NaN
2,85168.lin,open,14,E,None,QJ983 | KT2 | T75 | QT,K76 | 5 | QJ64 | AK752,AT542 | AJ6 | A983 | 4,- | Q98743 | K2 | J9863,1C 1S d 2N p 4S 5C p p d p p p,5Cx,E,0.0,11.0
3,85168.lin,closed,14,E,None,QJ983 | KT2 | T75 | QT,K76 | 5 | QJ64 | AK752,AT542 | AJ6 | A983 | 4,- | Q98743 | K2 | J9863,1C 1S d 2C p 4S 5C p p d p p p,5Cx,E,0.0,11.0
4,85168.lin,open,15,S,NS,AT873 | K42 | 93 | Q85,K4 | A983 | AJ87 | K63,652 | J65 | T62 | AJ94,QJ9 | QT7 | KQ54 | T72,p p p 1N p 3N p p p,3N,E,-1.0,8.0
5,85168.lin,closed,15,S,NS,AT873 | K42 | 93 | Q85,K4 | A983 | AJ87 | K63,652 | J65 | T62 | AJ94,QJ9 | QT7 | KQ54 | T72,p p p 1N p 3N p p p,3N,E,NaN,NaN
6,85168.lin,open,16,W,EW,KQ864 | 65 | 72 | J754,A97 | 72 | KJ953 | A63,J52 | AT843 | Q4 | QT9,T3 | KQJ9 | AT86 | K82,1D 1S 2S 3S p p 3N p p p,3N,E,NaN,NaN
7,85168.lin,closed,16,W,EW,KQ864 | 65 | 72 | J754,A97 | 72 | KJ953 | A63,J52 | AT843 | Q4 | QT9,T3 | KQJ9 | AT86 | K82,1D 2D d 3H d 3S 3N p p p,3N,E,0.0,9.0
8,85168.lin,open,17,N,None,J9832 | T852 | 974 | 8,74 | A73 | A853 | K764,Q6 | K964 | KQJ2 | J52,AKT5 | QJ | T6 | AQT93,p p 1D 2C p 2D p 2S p 3N p p p,3N,E,1.0,10.0
9,85168.lin,closed,17,N,None,J9832 | T852 | 974 | 8,74 | A73 | A853 | K764,Q6 | K964 | KQJ2 | J52,AKT5 | QJ | T6 | AQT93,p 1C 1D 1H p 1N p 3N p p p,3N,E,0.0,9.0


In [4]:
# Board yang sama dari sisi PBN
display(raw_df[raw_df['file'].str.lower().str.endswith('.pbn')].head(10))

,file,room,board,dealer,vul,North,East,South,West,lelang,kontrak,declarer,hasil,trik
66,angelfire_bermud00_52B1USNO.PBN,open,1|Brogeland|Saelensminde|Silverman|Wolfson,N,None,85 | AQ83 | 943 | 7542,KQT | 6 | 852 | KQJT96,J732 | JT972 | AJT7 | -,A964 | K54 | KQ6 | A83,Pass 1C X XX 1H 2C Pass 3NT Pass Pass 4H X Pass Pass Pass,4Hx,N,-2.0,8.0
67,angelfire_bermud00_52B1USNO.PBN,closed,1|Austberg|Helgemo|Mahmood|Rosenberg,N,None,85 | AQ83 | 943 | 7542,KQT | 6 | 852 | KQJT96,J732 | JT972 | AJT7 | -,A964 | K54 | KQ6 | A83,Pass 1C Pass 1S Pass 2C Pass 2D Pass 2S Pass 3S Pass 4H ...,6C,E,-1.0,11.0
68,angelfire_bermud00_52B1USNO.PBN,open,2|Brogeland|Saelensminde|Silverman|Wolfson,E,NS,QJT | QJ5 | KT432 | 53,A983 | T93 | Q85 | A42,K74 | K8762 | 6 | KT86,652 | A4 | AJ97 | QJ97,Pass Pass 1C 1D 1S X XX Pass 1NT Pass Pass Pass,1N,E,-1.0,6.0
69,angelfire_bermud00_52B1USNO.PBN,closed,2|Austberg|Helgemo|Mahmood|Rosenberg,E,NS,QJT | QJ5 | KT432 | 53,A983 | T93 | Q85 | A42,K74 | K8762 | 6 | KT86,652 | A4 | AJ97 | QJ97,Pass Pass 1NT Pass Pass 2C X 2D 2NT Pass Pass Pass,2N,W,-4.0,4.0
70,angelfire_bermud00_52B1USNO.PBN,open,3|Brogeland|Saelensminde|Silverman|Wolfson,S,EW,J64 | KT974 | 7 | KT92,A52 | 5 | J95432 | A43,T973 | 86 | KQT6 | J87,KQ8 | AQJ32 | A8 | Q65,Pass 1H Pass 1NT Pass 3NT Pass Pass Pass,3N,E,0.0,9.0
71,angelfire_bermud00_52B1USNO.PBN,closed,3|Austberg|Helgemo|Mahmood|Rosenberg,S,EW,J64 | KT974 | 7 | KT92,A52 | 5 | J95432 | A43,T973 | 86 | KQT6 | J87,KQ8 | AQJ32 | A8 | Q65,Pass 1H Pass 1NT Pass 2NT Pass 3NT Pass Pass Pass,3N,E,0.0,9.0
72,angelfire_bermud00_52B1USNO.PBN,open,4|Brogeland|Saelensminde|Silverman|Wolfson,W,Both,QJ9 | T4 | Q9432 | 864,764 | 8653 | J65 | A52,T | AKQJ92 | AKT7 | K9,AK8532 | 7 | 8 | QJT73,1S Pass Pass 2S 3C Pass 3S 4H Pass Pass 4S Pass Pass 5D ...,5H,S,0.0,11.0
73,angelfire_bermud00_52B1USNO.PBN,closed,4|Austberg|Helgemo|Mahmood|Rosenberg,W,Both,QJ9 | T4 | Q9432 | 864,764 | 8653 | J65 | A52,T | AKQJ92 | AKT7 | K9,AK8532 | 7 | 8 | QJT73,1S Pass 1NT 4H 4S Pass Pass X Pass Pass Pass,4Sx,W,-1.0,9.0
74,angelfire_bermud00_52B1USNO.PBN,open,5|Brogeland|Saelensminde|Silverman|Wolfson,N,NS,KQ9 | KT74 | A9 | AT52,T7532 | - | J854 | KQJ7,J64 | AQ9 | KT62 | 984,A8 | J86532 | Q73 | 63,1NT Pass 2D Pass 2H Pass 3NT Pass Pass Pass,3N,N,1.0,10.0
75,angelfire_bermud00_52B1USNO.PBN,closed,5|Austberg|Helgemo|Mahmood|Rosenberg,N,NS,KQ9 | KT74 | A9 | AT52,T7532 | - | J854 | KQJ7,J64 | AQ9 | KT62 | 984,A8 | J86532 | Q73 | 63,1NT Pass 3NT Pass Pass Pass,3N,N,1.0,10.0


---
## 2. Ukuran Dataset & Split

In [5]:
split_tbl = pd.DataFrame({
    'baris'           : [len(d) for d in (df_train, df_val, df_test, df)],
    'persen'          : [len(d) / len(df) * 100 for d in (df_train, df_val, df_test, df)],
    'matches_par = 1' : [d['matches_par_contract'].sum() for d in (df_train, df_val, df_test, df)],
    'rate_par (%)'    : [d['matches_par_contract'].mean() * 100 for d in (df_train, df_val, df_test, df)],
    'file sumber unik': [d['_source_file'].nunique() for d in (df_train, df_val, df_test, df)],
}, index=['train', 'val', 'test', 'full']).round(2)
display(split_tbl)

col_tbl = pd.DataFrame({
    'kelompok' : ['Fitur model', 'Metadata (_*)', 'Target / label', 'Par DDS (leakage, bukan fitur)'],
    'jumlah'   : [len(feature_cols), len(meta_cols), len(target_cols), len(leakage_cols)],
    'contoh'   : [', '.join(feature_cols[:4]) + ', ...', ', '.join(meta_cols),
                  ', '.join(target_cols), ', '.join(leakage_cols[:4]) + ', ...'],
})
display(col_tbl)

,baris,persen,matches_par = 1,rate_par (%),file sumber unik
train,35161,70.0,8635,24.56,1929
val,7534,15.0,1851,24.57,1669
test,7532,15.0,1849,24.55,1632
full,50227,100.0,12335,24.56,1930


,kelompok,jumlah,contoh
0,Fitur model,174,"N_hcp, N_hcp_S, N_len_S, N_stopper_S, ..."
1,Metadata (_*),7,"_board_number, _source_file, _room, _declarer, _result, ..."
2,Target / label,5,"matches_par_contract, label, target_base, target, target..."
3,"Par DDS (leakage, bukan fitur)",9,"dd_par_level, dd_par_declarer_is_ns, dd_par_score, dd_pa..."


---
## 3. `head` / `tail` / `sample`

174 fitur terlalu lebar untuk satu tabel, jadi ditampilkan per kelompok.

In [6]:
key_cols = ['ns_hcp', 'ew_hcp', 'ns_hcp_advantage', 'ns_best_fit', 'ew_best_fit',
            'ns_dd_N', 'ns_dd_S', 'ew_dd_N', 'auction_len', 'opening_level']
key_cols = [c for c in key_cols if c in df.columns]
overview_cols = meta_cols + target_cols + key_cols

print('=== head(10): metadata + target + fitur kunci ===')
display(df[overview_cols].head(10))

=== head(10): metadata + target + fitur kunci ===


,_board_number,_source_file,_room,_declarer,_result,_tricks_made,_deal_key,matches_par_contract,label,target_base,target,target_category,ns_hcp,ew_hcp,ns_hcp_advantage,ns_best_fit,ew_best_fit,ns_dd_N,ns_dd_S,ew_dd_N,auction_len,opening_level
0,13,85168.lin,open,W,-1.0,8.0,"N:SKT87,HT76,D873,C743|E:SQ642,HJ532,D2,CAQ62|S:SJ9,HAQ9...",0,0,3N,3N,Game,14,26,-12,8,7,4.0,3.0,9.0,15,1
1,13,85168.lin,closed,W,NaN,NaN,"N:SKT87,HT76,D873,C743|E:SQ642,HJ532,D2,CAQ62|S:SJ9,HAQ9...",0,0,3N,3N,Game,14,26,-12,8,7,4.0,3.0,9.0,13,1
2,14,85168.lin,open,E,0.0,11.0,"N:SQJ983,HKT2,DT75,CQT|E:SK76,H5,DQJ64,CAK752|S:SAT542,H...",0,0,5C,5Cx,Game,21,19,2,10,10,8.0,10.0,4.0,13,1
3,14,85168.lin,closed,E,0.0,11.0,"N:SQJ983,HKT2,DT75,CQT|E:SK76,H5,DQJ64,CAK752|S:SAT542,H...",0,0,5C,5Cx,Game,21,19,2,10,10,8.0,10.0,4.0,13,1
4,15,85168.lin,open,E,-1.0,8.0,"N:SAT873,HK42,D93,CQ85|E:SK4,HA983,DAJ87,CK63|S:S652,HJ6...",0,0,3N,3N,Game,15,25,-10,8,8,4.0,6.0,8.0,9,1
5,16,85168.lin,open,E,NaN,NaN,"N:SKQ864,H65,D72,CJ754|E:SA97,H72,DKJ953,CA63|S:SJ52,HAT...",1,1,3N,3N,Game,15,25,-10,8,9,2.0,6.0,10.0,10,1
6,16,85168.lin,closed,E,0.0,9.0,"N:SKQ864,H65,D72,CJ754|E:SA97,H72,DKJ953,CA63|S:SJ52,HAT...",1,1,3N,3N,Game,15,25,-10,8,9,2.0,6.0,10.0,10,1
7,17,85168.lin,open,E,1.0,10.0,"N:SJ9832,HT852,D974,C8|E:S74,HA73,DA853,CK764|S:SQ6,HK96...",1,1,3N,3N,Game,13,27,-14,8,9,3.0,5.0,10.0,13,1
8,17,85168.lin,closed,E,0.0,9.0,"N:SJ9832,HT852,D974,C8|E:S74,HA73,DA853,CK764|S:SQ6,HK96...",1,1,3N,3N,Game,13,27,-14,8,9,3.0,5.0,10.0,11,1
9,18,85168.lin,open,N,0.0,12.0,"N:SAT98,H7532,DAKQ2,CA|E:S64,HQ4,DJ653,CQJ976|S:SKJ72,HA...",0,0,6H,6H,SmallSlam,28,12,16,8,8,10.0,13.0,2.0,19,1


In [7]:
print('=== tail(5) ===')
display(df[overview_cols].tail(5))
print('=== sample(10, random_state=42) ===')
display(df[overview_cols].sample(10, random_state=42))

=== tail(5) ===


,_board_number,_source_file,_room,_declarer,_result,_tricks_made,_deal_key,matches_par_contract,label,target_base,target,target_category,ns_hcp,ew_hcp,ns_hcp_advantage,ns_best_fit,ew_best_fit,ns_dd_N,ns_dd_S,ew_dd_N,auction_len,opening_level
50222,30|CLEMENTSSO|OVELIUS|Q SHEN|WEN WANG|,venice_cup_2019_66005.pbn,closed,S,-1.0,8.0,"N:SQJ952,HQ4,D9,CKQJ64|E:SAK763,HK9763,DQ,C83|S:S8,HAJ8,...",1,1,3D,3Dx,Partscore,19,21,-2,8,8,7.0,6.0,6.0,10,1
50223,31|C RIMSTEDT|GRONKVIST|YA LU|YAN LIU,venice_cup_2019_66005.pbn,open,W,-1.0,8.0,"N:SQT6,HQ4,DA732,CA642|E:SK72,HAT,DKQJT54,C95|S:S53,HKJ8...",0,0,3S,3S,Partscore,22,18,4,9,8,6.0,4.0,5.0,13,1
50224,31|CLEMENTSSO|OVELIUS|Q SHEN|WEN WANG|,venice_cup_2019_66005.pbn,closed,N,0.0,10.0,"N:SQT6,HQ4,DA732,CA642|E:SK72,HAT,DKQJT54,C95|S:S53,HKJ8...",0,0,4C,4C,Partscore,22,18,4,9,8,6.0,4.0,5.0,18,1
50225,32|C RIMSTEDT|GRONKVIST|YA LU|YAN LIU,venice_cup_2019_66005.pbn,open,S,0.0,12.0,"N:SJ942,HT86,D53,CT982|E:S65,HKJ543,DT962,C73|S:SAKQT3,H...",0,0,6C,6C,SmallSlam,24,16,8,10,10,7.0,12.0,1.0,7,1
50226,32|CLEMENTSSO|OVELIUS|Q SHEN|WEN WANG|,venice_cup_2019_66005.pbn,closed,N,0.0,12.0,"N:SJ942,HT86,D53,CT982|E:S65,HKJ543,DT962,C73|S:SAKQT3,H...",1,1,6S,6S,SmallSlam,24,16,8,10,10,7.0,12.0,1.0,19,1


=== sample(10, random_state=42) ===


,_board_number,_source_file,_room,_declarer,_result,_tricks_made,_deal_key,matches_par_contract,label,target_base,target,target_category,ns_hcp,ew_hcp,ns_hcp_advantage,ns_best_fit,ew_best_fit,ns_dd_N,ns_dd_S,ew_dd_N,auction_len,opening_level
11562,16,TM-48595-1751633991.lin,closed,N,NaN,NaN,"N:SKQJT64,H97,DT64,C75|E:S8,HKJT843,D7,CAKJT3|S:S75,HA62...",0,0,3S,3S,Partscore,18,22,-4,8,8,4.0,8.0,7.0,9,2
2322,11,85355.lin,open,E,0.0,10.0,"N:SAT632,H93,DKQ7,CQ94|E:SK,HAQT52,DA543,CAK3|S:SQ75,H64...",1,1,4H,4H,Game,14,26,-12,8,9,4.0,6.0,8.0,13,1
7427,26,86309.lin,closed,N,NaN,NaN,"N:SA65,HT2,DKJT83,CAKQ|E:SKQJ,H986,DA52,C7653|S:S8432,HK...",0,0,1N,1N,Partscore,21,19,2,7,7,6.0,6.0,7.0,7,1
26068,5|Brachman|Fu|Ju|Passell,angelfire_forbo00_N1BCHIUS.PBN,open,S,2.0,11.0,"N:SAKQJ6,HT952,DKJ,CQ5|E:S9843,HAQ63,D32,CAJ2|S:ST,HJ8,D...",1,1,3N,3N,Game,24,16,8,7,7,9.0,9.0,4.0,12,1
49176,731|Bigguy46|POOHBYR|papi|snabu,tistis_papi_imp_13.pbn,NaN,N,-1.0,7.0,"N:S97,HAJ973,DT65,CAQ8|E:SK85,HK,DA8732,CK653|S:SQJT62,H...",0,0,2H,2H,Partscore,17,23,-6,8,8,4.0,6.0,8.0,8,1
9403,9,86426.lin,closed,S,NaN,NaN,"N:SJ92,H42,D73,CAK9542|E:SKT6,HQJ6,DQ9652,CQ3|S:SA43,HAK...",0,0,3C,3C,Partscore,22,18,4,8,8,6.0,6.0,5.0,6,2
20419,18|M Passell|M Se|Robert levin|Steve Weinstein,angelfire_cavend00_TEAMS.PBN,closed,NaN,NaN,NaN,"N:SKT5,H982,DQJ3,CKJ42|E:SQ93,HAQJT,DA986,CT6|S:SA8642,H...",0,0,PASS,PASS,Pass,21,19,2,8,9,7.0,8.0,5.0,0,0
36294,23|Denmark|Belgium,angelfire_montec97_OPDANBEL.PBN,open,W,1.0,9.0,"N:SQJT73,HA7,DQ6,CA983|E:SK96,HK984,DAK75,CQ7|S:S42,HJ32...",0,0,2H,2H,Partscore,17,23,-6,9,8,6.0,8.0,7.0,9,1
11682,14,TM-53609-1756124658.lin,open,N,2.0,12.0,"N:SKQT,HAKJ953,D7,CT63|E:S983,H42,DJT984,CAQJ|S:SA42,HT8...",1,1,4H,4H,Game,26,14,12,9,8,9.0,9.0,1.0,13,1
6976,15,86284.lin,open,E,1.0,11.0,"N:S2,HJT987,DQ952,C943|E:SAJ8764,HA,DJ8,CAT65|S:ST93,HKQ...",1,1,4S,4S,Game,15,25,-10,9,9,2.0,2.0,9.0,11,1


In [8]:
col_groups = {
    'Per-seat North (N_*)'  : [c for c in feature_cols if c.startswith('N_')],
    'Per-seat East (E_*)'   : [c for c in feature_cols if c.startswith('E_')],
    'Per-seat South (S_*)'  : [c for c in feature_cols if c.startswith('S_')],
    'Per-seat West (W_*)'   : [c for c in feature_cols if c.startswith('W_')],
    'Partnership NS (ns_*)' : [c for c in feature_cols if c.startswith('ns_') and '_dd_' not in c],
    'Partnership EW (ew_*)' : [c for c in feature_cols if c.startswith('ew_') and '_dd_' not in c],
    'Deal context'          : [c for c in feature_cols if c.startswith(('dealer_', 'vuln_'))],
    'Auction'               : [c for c in feature_cols if c.startswith(('auction_', 'opening_'))],
    'Trik DD (fitur)'       : [c for c in feature_cols if '_dd_' in c],
    'Par DDS (leakage)'     : leakage_cols,
}
assert sum(len(v) for k, v in col_groups.items() if k != 'Par DDS (leakage)') == len(feature_cols), \
    'Ada fitur yang tidak masuk kelompok mana pun'

for name, cols in col_groups.items():
    print(f'\n=== {name}  ({len(cols)} kolom) ===')
    display(df[cols].head(5))


=== Per-seat North (N_*)  (24 kolom) ===


,N_hcp,N_hcp_S,N_len_S,N_stopper_S,N_hcp_H,N_len_H,N_stopper_H,N_hcp_D,N_len_D,N_stopper_D,N_hcp_C,N_len_C,N_stopper_C,N_controls,N_balanced,N_voids,N_singletons,N_doubletons,N_ltc,N_longest_suit_len,N_is_longest_S,N_is_longest_H,N_is_longest_D,N_is_longest_C
0,3,3,4,1,0,3,0,0,3,0,0,3,0,1,1,0,0,0,11,4,1,0,0,0
1,3,3,4,1,0,3,0,0,3,0,0,3,0,1,1,0,0,0,11,4,1,0,0,0
2,8,3,5,1,3,3,1,0,3,0,2,2,0,1,1,0,0,1,8,5,1,0,0,0
3,8,3,5,1,3,3,1,0,3,0,2,2,0,1,1,0,0,1,8,5,1,0,0,0
4,9,4,5,1,3,3,1,0,2,0,2,3,1,3,1,0,0,1,8,5,1,0,0,0



=== Per-seat East (E_*)  (24 kolom) ===


,E_hcp,E_hcp_S,E_len_S,E_stopper_S,E_hcp_H,E_len_H,E_stopper_H,E_hcp_D,E_len_D,E_stopper_D,E_hcp_C,E_len_C,E_stopper_C,E_controls,E_balanced,E_voids,E_singletons,E_doubletons,E_ltc,E_longest_suit_len,E_is_longest_S,E_is_longest_H,E_is_longest_D,E_is_longest_C
0,9,2,4,1,1,4,1,0,1,0,6,4,1,2,0,0,1,0,7,4,0,0,0,1
1,9,2,4,1,1,4,1,0,1,0,6,4,1,2,0,0,1,0,7,4,0,0,0,1
2,13,3,3,1,0,1,0,3,4,1,7,5,1,4,0,0,1,0,6,5,0,0,0,1
3,13,3,3,1,0,1,0,3,4,1,7,5,1,4,0,0,1,0,6,5,0,0,0,1
4,15,3,2,1,4,4,1,5,4,1,3,3,1,6,1,0,0,1,7,4,0,0,1,0



=== Per-seat South (S_*)  (24 kolom) ===


...

,auction_len,auction_ns_bids,auction_ew_bids,auction_competitive,auction_has_double,auction_doubled,auction_redoubled,opening_level,opening_strain_C,opening_strain_D,opening_strain_H,opening_strain_S,opening_strain_N,opening_strain_PASS,auction_alerts
0,15,1,5,1,0,0,0,1,0,1,0,0,0,0,0
1,13,1,4,1,0,0,0,1,0,1,0,0,0,0,0
2,13,3,2,1,1,1,0,1,1,0,0,0,0,0,1
3,13,3,2,1,1,1,0,1,1,0,0,0,0,0,0
4,9,0,2,0,0,0,0,1,0,0,0,0,1,0,0



=== Trik DD (fitur)  (10 kolom) ===


,ns_dd_S,ew_dd_S,ns_dd_H,ew_dd_H,ns_dd_D,ew_dd_D,ns_dd_C,ew_dd_C,ns_dd_N,ew_dd_N
0,3.0,10.0,3.0,10.0,6.0,7.0,3.0,10.0,4.0,9.0
1,3.0,10.0,3.0,10.0,6.0,7.0,3.0,10.0,4.0,9.0
2,10.0,3.0,5.0,8.0,7.0,5.0,2.0,11.0,8.0,4.0
3,10.0,3.0,5.0,8.0,7.0,5.0,2.0,11.0,8.0,4.0
4,6.0,7.0,4.0,9.0,4.0,9.0,6.0,7.0,4.0,8.0



=== Par DDS (leakage)  (9 kolom) ===


,dd_par_level,dd_par_declarer_is_ns,dd_par_score,dd_par_contracts,dd_par_denom_S,dd_par_denom_H,dd_par_denom_D,dd_par_denom_C,dd_par_denom_N
0,4.0,0.0,-620.0,4H@EW|4S@EW,1.0,0.0,0.0,0.0,0.0
1,4.0,0.0,-620.0,4H@EW|4S@EW,1.0,0.0,0.0,0.0,0.0
2,5.0,1.0,-100.0,5Sx@NS,1.0,0.0,0.0,0.0,0.0
3,5.0,1.0,-100.0,5Sx@NS,1.0,0.0,0.0,0.0,0.0
4,1.0,0.0,-140.0,1H@EW,0.0,1.0,0.0,0.0,0.0


---
## 4. Kamus Data

Satu baris per kolom: kelompok, tipe data, jumlah non-null/missing, jumlah
nilai unik, min/max/mean (kolom numerik), dan contoh nilai. Disimpan ke
`outputs/results/nb01b_data_dictionary.csv`.

In [9]:
def col_group(c):
    if c.startswith('_'):
        return 'metadata'
    if c in target_cols:
        return 'target'
    if c in leakage_cols:
        return 'par DDS (leakage)'
    for name, cols in col_groups.items():
        if c in cols:
            return name
    return 'lainnya'

num = df.select_dtypes(include='number')
data_dict = pd.DataFrame({
    'kelompok' : [col_group(c) for c in df.columns],
    'dtype'    : df.dtypes.astype(str).values,
    'non_null' : df.notna().sum().values,
    'missing'  : df.isna().sum().values,
    'n_unique' : df.nunique().values,
    'min'      : [num[c].min() if c in num else None for c in df.columns],
    'max'      : [num[c].max() if c in num else None for c in df.columns],
    'mean'     : [round(num[c].mean(), 3) if c in num else None for c in df.columns],
    'contoh'   : [', '.join(map(str, df[c].dropna().unique()[:4])) for c in df.columns],
}, index=pd.Index(df.columns, name='kolom'))

data_dict.to_csv(OUT / 'nb01b_data_dictionary.csv')
print('Ringkasan tipe data:')
display(data_dict.groupby(['kelompok', 'dtype']).size().rename('jumlah kolom').to_frame())

Ringkasan tipe data:


,,jumlah kolom
kelompok,dtype,
Auction,int64,15
Deal context,int64,8
Partnership EW (ew_*),int64,22
Partnership NS (ns_*),int64,23
Per-seat East (E_*),int64,24
Per-seat North (N_*),int64,24
Per-seat South (S_*),int64,24
Per-seat West (W_*),int64,24
Trik DD (fitur),float64,10


In [10]:
# Kamus lengkap (semua kolom)
with pd.option_context('display.max_rows', None):
    display(data_dict)

,kelompok,dtype,non_null,missing,n_unique,min,max,mean,contoh
kolom,,,,,,,,,
N_hcp,Per-seat North (N_*),int64,50227,0,28,0.0,27.0,10.012,"3, 8, 9, 6"
N_hcp_S,Per-seat North (N_*),int64,50227,0,11,0.0,10.0,2.510,"3, 4, 5, 1"
N_len_S,Per-seat North (N_*),int64,50227,0,10,0.0,9.0,3.260,"4, 5, 1, 2"
N_stopper_S,Per-seat North (N_*),int64,50227,0,2,0.0,1.0,0.594,"1, 0"
N_hcp_H,Per-seat North (N_*),int64,50227,0,11,0.0,10.0,2.499,"0, 3, 2, 6"
N_len_H,Per-seat North (N_*),int64,50227,0,10,0.0,9.0,3.237,"3, 2, 4, 5"
N_stopper_H,Per-seat North (N_*),int64,50227,0,2,0.0,1.0,0.587,"0, 1"
N_hcp_D,Per-seat North (N_*),int64,50227,0,11,0.0,10.0,2.482,"0, 9, 4, 5"
N_len_D,Per-seat North (N_*),int64,50227,0,10,0.0,9.0,3.238,"3, 2, 4, 5"


---
## 5. Statistik Deskriptif (`describe`)

In [11]:
desc_cols = ['ns_hcp', 'ew_hcp', 'ns_hcp_advantage', 'ns_ltc', 'ew_ltc',
             'ns_controls', 'ew_controls', 'ns_best_fit', 'ew_best_fit', 'ns_nt_stoppers',
             'ns_dd_S', 'ns_dd_H', 'ns_dd_D', 'ns_dd_C', 'ns_dd_N',
             'auction_len', 'auction_ns_bids', 'auction_ew_bids', 'opening_level', 'auction_alerts']
desc_cols = [c for c in desc_cols if c in df.columns]

print('=== Fitur numerik kunci ===')
display(df[desc_cols].describe().T.round(2))

=== Fitur numerik kunci ===


,count,mean,std,min,25%,50%,75%,max
ns_hcp,50227.0,20.06,4.81,2.0,17.0,20.0,23.0,36.0
ew_hcp,50227.0,19.94,4.81,4.0,17.0,20.0,23.0,38.0
ns_hcp_advantage,50227.0,0.12,9.61,-36.0,-6.0,0.0,6.0,32.0
ns_ltc,50227.0,14.74,2.09,6.0,13.0,15.0,16.0,22.0
ew_ltc,50227.0,14.77,2.08,7.0,13.0,15.0,16.0,22.0
ns_controls,50227.0,6.02,2.11,0.0,5.0,6.0,8.0,12.0
ew_controls,50227.0,5.98,2.11,0.0,4.0,6.0,7.0,12.0
ns_best_fit,50227.0,8.37,0.91,7.0,8.0,8.0,9.0,12.0
ew_best_fit,50227.0,8.36,0.90,7.0,8.0,8.0,9.0,13.0
ns_nt_stoppers,50227.0,0.58,0.49,0.0,0.0,1.0,1.0,1.0


In [12]:
print('=== Per-seat: HCP, LTC, panjang suit terpanjang ===')
seat_cols = [f'{s}_{f}' for f in ('hcp', 'ltc', 'longest_suit_len') for s in 'NESW'
             if f'{s}_{f}' in df.columns]
display(df[seat_cols].describe().T.round(2))

print('=== Kolom kategorikal (object) ===')
display(df.select_dtypes(include='object').describe().T)

=== Per-seat: HCP, LTC, panjang suit terpanjang ===


,count,mean,std,min,25%,50%,75%,max
N_hcp,50227.0,10.01,4.13,0.0,7.0,10.0,13.0,27.0
E_hcp,50227.0,9.93,4.16,0.0,7.0,10.0,13.0,30.0
S_hcp,50227.0,10.05,4.15,0.0,7.0,10.0,13.0,28.0
W_hcp,50227.0,10.01,4.13,0.0,7.0,10.0,13.0,28.0
N_ltc,50227.0,7.37,1.64,0.0,6.0,7.0,8.0,12.0
E_ltc,50227.0,7.40,1.65,1.0,6.0,7.0,9.0,12.0
S_ltc,50227.0,7.37,1.64,1.0,6.0,7.0,8.0,12.0
W_ltc,50227.0,7.37,1.63,1.0,6.0,7.0,8.0,12.0
N_longest_suit_len,50227.0,4.91,0.84,4.0,4.0,5.0,5.0,9.0
E_longest_suit_len,50227.0,4.91,0.83,4.0,4.0,5.0,5.0,10.0


=== Kolom kategorikal (object) ===


,count,unique,top,freq
_board_number,50227,31173,3,557
_source_file,50227,1930,tistis_papi_imp_01.pbn,193
_room,41493,2,open,23022
_declarer,49932,9,S,12781
target,50227,88,3N,9253
target_base,50227,36,3N,9403
target_category,50227,5,Game,26224
_deal_key,50227,22792,"N:SQT87,HQT7653,D,CAT9|E:SAK42,HAK98,DA93,CQ7|S:S953,HJ4...",16
dd_par_contracts,50227,369,3N@NS,3601


---
## 6. Kualitas Data: Missing Value & Duplikat

In [13]:
missing = df.isna().sum()
missing = missing[missing > 0].sort_values(ascending=False)
miss_tbl = pd.DataFrame({
    'kelompok': [col_group(c) for c in missing.index],
    'missing' : missing,
    'persen'  : (missing / len(df) * 100).round(2),
})
print('=== Kolom dengan missing value ===')
display(miss_tbl if not miss_tbl.empty else pd.DataFrame({'info': ['tidak ada missing value']}))

quality = pd.Series({
    f'Missing di {len(feature_cols)} fitur model'   : int(df[feature_cols].isna().sum().sum()),
    'Baris duplikat penuh'                          : int(df.duplicated().sum()),
    'Baris duplikat pada fitur model'               : int(df.duplicated(subset=feature_cols).sum()),
    'Fitur model non-numerik'                       : sum(not pd.api.types.is_numeric_dtype(df[c]) for c in feature_cols),
}, name='jumlah').to_frame()
display(quality)

=== Kolom dengan missing value ===

,kelompok,missing,persen
_room,metadata,8734,17.39
_result,metadata,3942,7.85
_tricks_made,metadata,3942,7.85
_declarer,metadata,295,0.59


,jumlah
Missing di 174 fitur model,0
Baris duplikat penuh,0
Baris duplikat pada fitur model,472
Fitur model non-numerik,0


In [14]:
# Nilai _declarer di luar N/E/S/W (sisa token parser, hanya metadata -- tidak dipakai model)
odd_decl = df[~df['_declarer'].isin(list('NESW'))]
print(f'Baris dengan _declarer di luar N/E/S/W: {len(odd_decl):,}')
display(odd_decl['_declarer'].value_counts(dropna=False).rename('jumlah').to_frame())
display(odd_decl[overview_cols].head(10))

Baris dengan _declarer di luar N/E/S/W: 302


,jumlah
_declarer,
NaN,295
^E,2
^S,2
?,1
^N,1
^W,1


,_board_number,_source_file,_room,_declarer,_result,_tricks_made,_deal_key,matches_par_contract,label,target_base,target,target_category,ns_hcp,ew_hcp,ns_hcp_advantage,ns_best_fit,ew_best_fit,ns_dd_N,ns_dd_S,ew_dd_N,auction_len,opening_level
328,10,85205.lin,closed,NaN,NaN,NaN,"N:S6,HA9,DAQ9632,CQ742|E:SKQT8,HJT753,D8,CKJ9|S:SAJ954,H...",0,0,PASS,PASS,Pass,21,19,2,9,8,9.0,7.0,4.0,4,0
353,20,85207.lin,closed,NaN,NaN,NaN,"N:SAT982,H9,DK84,CAKT3|E:SKQ7,HAK6,DJ732,CQJ6|S:S4,HQJ87...",0,0,PASS,PASS,Pass,23,17,6,8,7,8.0,8.0,5.0,4,0
735,11,85234.lin,closed,NaN,NaN,NaN,"N:S64,HKQJ92,DKT4,CJ76|E:SAT8,HT75,DAJ,CAKQ95|S:SJ9753,H...",0,0,PASS,PASS,Pass,17,23,-6,7,7,4.0,5.0,9.0,4,0
1298,26,85280.lin,open,NaN,NaN,NaN,"N:SQ3,HJT8,DQ9,CKQJ875|E:SA74,HQ9765,D854,CA4|S:SJ9862,H...",0,0,PASS,PASS,Pass,22,18,4,8,8,6.0,8.0,4.0,4,0
1746,24,85322.lin,closed,NaN,NaN,NaN,"N:SAKT4,HT43,DKJT,CT93|E:SQ92,HQJ65,DQ432,CQ5|S:SJ7,HAK9...",0,0,PASS,PASS,Pass,22,18,4,8,8,8.0,7.0,5.0,4,0
3263,25,85439.lin,closed,NaN,NaN,NaN,"N:ST,HK3,DAQJ9542,CQJ3|E:SK93,HA985,D63,CT987|S:S7652,HQ...",0,0,PASS,PASS,Pass,18,22,-4,10,8,6.0,4.0,6.0,3,0
3703,32,85483.lin,closed,NaN,NaN,NaN,"N:ST9754,HAQ6,DT5,CAT3|E:SAQ86,HT532,DQ6,CK85|S:SKJ32,HK...",0,0,PASS,PASS,Pass,19,21,-2,9,9,5.0,8.0,8.0,4,0
4249,10,85605.lin,open,NaN,NaN,NaN,"N:SAQ4,HQ73,DQ432,CJT9|E:ST93,HA8654,DAJ,CQ82|S:SK7,HKT9...",0,0,PASS,PASS,Pass,20,20,0,7,8,8.0,4.0,5.0,4,0
4422,25,85627.lin,closed,NaN,NaN,NaN,"N:SAQT76,HK,DJT95,C984|E:S54,HJ432,DA432,CAJ7|S:SK983,HA...",0,0,PASS,PASS,Pass,19,21,-2,9,8,6.0,9.0,5.0,4,0
4457,1,85630.lin,closed,NaN,NaN,NaN,"N:SA96,H9754,D43,CK975|E:S83,HAJ3,DQ9862,CA62|S:SQT54,HK...",0,0,PASS,PASS,Pass,17,23,-6,8,9,5.0,6.0,8.0,4,0


---
## 7. Tabel Frekuensi Kolom Kategorikal

In [15]:
def freq(s, name=None):
    vc = s.value_counts(dropna=False)
    return pd.DataFrame({'jumlah': vc, 'persen': (vc / len(s) * 100).round(2)}).rename_axis(name or s.name)

display(freq(source_type, 'sumber'))
display(freq(df['_room'], '_room'))
display(freq(df['_declarer'].where(df['_declarer'].isin(list('NESW')), 'lainnya/NaN'), '_declarer'))

,jumlah,persen
sumber,,
PBN,37756,75.17
LIN (BBO),12471,24.83


,jumlah,persen
_room,,
open,23022,45.84
closed,18471,36.78
NaN,8734,17.39


,jumlah,persen
_declarer,,
S,12781,25.45
N,12776,25.44
W,12272,24.43
E,12096,24.08
lainnya/NaN,302,0.60


In [16]:
# Dealer & vulnerability tersimpan sebagai one-hot -> gabungkan jadi satu kategori
dealer = df[['dealer_N', 'dealer_E', 'dealer_S', 'dealer_W']].idxmax(axis=1).str.replace('dealer_', '')
vuln   = df[['vuln_none', 'vuln_ns', 'vuln_ew', 'vuln_both']].idxmax(axis=1).str.replace('vuln_', '')
print('=== Dealer x Vulnerability ===')
display(pd.crosstab(dealer.rename('dealer'), vuln.rename('vuln'), margins=True))

opening = df[[c for c in df.columns if c.startswith('opening_strain_')]].idxmax(axis=1).str.replace('opening_strain_', '')
print('=== Opening level x Opening strain ===')
display(pd.crosstab(df['opening_level'].rename('opening_level'), opening.rename('opening_strain'), margins=True))

=== Dealer x Vulnerability ===


vuln,both,ew,none,ns,All
dealer,,,,,
E,2363,3249,3309,3817,12738
N,2087,3037,4461,3254,12839
S,2484,3846,3627,2361,12318
W,2955,2308,4177,2892,12332
All,9889,12440,15574,12324,50227


=== Opening level x Opening strain ===


opening_strain,C,D,H,N,PASS,S,All
opening_level,,,,,,,
0,0,0,0,0,931,0,931
1,11231,9592,6927,6744,0,6836,41330
2,1512,1755,1122,744,0,1053,6186
3,378,350,242,87,0,237,1294
4,49,32,185,1,0,144,411
5,24,23,4,0,0,3,54
6,4,2,3,3,0,5,17
7,0,1,0,2,0,1,4
All,13198,11755,8483,7581,931,8279,50227


In [17]:
print('=== target_category ===')
display(freq(df['target_category']))

print('=== target_base (kontrak level+strain), 15 teratas ===')
display(freq(df['target_base']).head(15))

print('=== target_base sebagai tabel level x strain ===')
tb = df.loc[df['target_base'] != 'PASS', 'target_base']
display(pd.crosstab(tb.str[0].rename('level'), tb.str[1].rename('strain'))[['C', 'D', 'H', 'S', 'N']])

=== target_category ===


,jumlah,persen
target_category,,
Game,26224,52.21
Partscore,20059,39.94
SmallSlam,3233,6.44
GrandSlam,415,0.83
Pass,296,0.59


=== target_base (kontrak level+strain), 15 teratas ===


,jumlah,persen
target_base,,
3N,9403,18.72
4S,7608,15.15
4H,6530,13.00
2S,2959,5.89
1N,2638,5.25
2H,2060,4.10
3S,2011,4.00
3H,1843,3.67
3D,1599,3.18


=== target_base sebagai tabel level x strain ===


strain,C,D,H,S,N
level,,,,,
1,33,80,154,321,2638
2,542,949,2060,2959,921
3,1492,1599,1843,2011,9403
4,500,454,6530,7608,210
5,1286,1397,749,524,20
6,568,576,828,897,364
7,58,68,126,96,67


---
## 8. Distribusi Target `matches_par_contract`

In [18]:
tgt = pd.DataFrame({
    split: d['matches_par_contract'].value_counts().sort_index()
    for split, d in [('train', df_train), ('val', df_val), ('test', df_test), ('full', df)]
})
tgt.index = ['Tidak cocok par (0)', 'Cocok par (1)']
tgt_pct = (tgt / tgt.sum() * 100).round(2).add_suffix(' (%)')
display(pd.concat([tgt, tgt_pct], axis=1))

,train,val,test,full,train (%),val (%),test (%),full (%)
Tidak cocok par (0),26526,5683,5683,37892,75.44,75.43,75.45,75.44
Cocok par (1),8635,1851,1849,12335,24.56,24.57,24.55,24.56


In [19]:
def rate_by(key, name):
    return (df.groupby(key)['matches_par_contract']
              .agg(n='size', cocok_par='sum', rate_par='mean')
              .assign(rate_par=lambda t: (t['rate_par'] * 100).round(2))
              .rename_axis(name)
              .sort_values('n', ascending=False))

display(rate_by(source_type, 'sumber'))
display(rate_by(df['target_category'], 'target_category'))
display(rate_by(vuln.rename('vuln'), 'vuln'))
display(rate_by(df['target_base'], 'target_base').head(15))

,n,cocok_par,rate_par
sumber,,,
PBN,37756,9300,24.63
LIN (BBO),12471,3035,24.34


,n,cocok_par,rate_par
target_category,,,
Game,26224,9148,34.88
Partscore,20059,2230,11.12
SmallSlam,3233,824,25.49
GrandSlam,415,133,32.05
Pass,296,0,0.00


,n,cocok_par,rate_par
vuln,,,
none,15574,3776,24.25
ew,12440,3002,24.13
ns,12324,3000,24.34
both,9889,2557,25.86


,n,cocok_par,rate_par
target_base,,,
3N,9403,3785,40.25
4S,7608,2674,35.15
4H,6530,2271,34.78
2S,2959,425,14.36
1N,2638,397,15.05
2H,2060,266,12.91
3S,2011,173,8.60
3H,1843,220,11.94
3D,1599,139,8.69


---
## 9. Perbandingan Fitur per Kelas Target

In [20]:
grp = df.groupby('matches_par_contract')[desc_cols].mean().T
grp.columns = ['Tidak cocok par (0)', 'Cocok par (1)']
grp['selisih (1-0)'] = grp['Cocok par (1)'] - grp['Tidak cocok par (0)']
display(grp.round(3).sort_values('selisih (1-0)', key=abs, ascending=False))

,Tidak cocok par (0),Cocok par (1),selisih (1-0)
auction_len,11.294,11.766,0.473
ns_ltc,14.689,14.896,0.207
ew_ltc,14.726,14.924,0.198
auction_ew_bids,2.163,2.292,0.129
auction_ns_bids,2.253,2.380,0.128
ns_best_fit,8.397,8.284,-0.114
ew_best_fit,8.392,8.278,-0.113
ns_hcp_advantage,0.098,0.183,0.085
ns_dd_N,6.036,6.085,0.049
ew_hcp,19.951,19.908,-0.043


In [21]:
# Contoh baris masing-masing kelas
for cls, label in [(1, 'Cocok par (1)'), (0, 'Tidak cocok par (0)')]:
    print(f'=== {label}: 5 contoh ===')
    display(df.loc[df['matches_par_contract'] == cls,
                   meta_cols + ['target_base', 'dd_par_level'] + leakage_cols[1:] + key_cols]
              .sample(5, random_state=42))

=== Cocok par (1): 5 contoh ===


,_board_number,_source_file,_room,_declarer,_result,_tricks_made,_deal_key,target_base,dd_par_level,dd_par_declarer_is_ns,dd_par_score,dd_par_contracts,dd_par_denom_S,dd_par_denom_H,dd_par_denom_D,dd_par_denom_C,dd_par_denom_N,ns_hcp,ew_hcp,ns_hcp_advantage,ns_best_fit,ew_best_fit,ns_dd_N,ns_dd_S,ew_dd_N,auction_len,opening_level
49911,840|Papi|randm|randme|snabu,tistis_papi_imp_21.pbn,NaN,W,0.0,9.0,"N:SKQT63,HKQ7,DK52,CQ8|E:S42,HAT62,DQ93,CKJT5|S:SJ875,HJ...",3C,3.0,0.0,-130.0,3C@EW|3D@EW,0.0,0.0,1.0,0.0,0.0,18,22,-4,9,9,3.0,7.0,8.0,16,1
38629,1|Balicki|Ingimarsson|Magnusson|Zmudzinski,angelfire_olymp00_Q1POLICE.PBN,closed,W,0.0,12.0,"N:S7654,H52,DK98,CAQT4|E:SKQ92,HK864,DA754,C9|S:SJT83,H7...",6H,6.0,0.0,-980.0,6H@EW,0.0,1.0,0.0,0.0,0.0,14,26,-12,8,9,3.0,5.0,9.0,25,1
8922,1,86406.lin,closed,S,NaN,NaN,"N:SQ962,H96,DQT7,CA984|E:ST753,HAQT3,DJ54,CJ2|S:SJ,HKJ54...",1N,1.0,1.0,120.0,1N@NS,0.0,0.0,0.0,0.0,1.0,19,21,-2,7,8,8.0,7.0,5.0,10,1
17345,5|Gitelman|Kwiecien|Pszczola|Stansby,angelfire_capgem02_round7.pbn,NaN,E,1.0,11.0,"N:SK2,HJT7,D74,CJ87654|E:SQT,HK9865,DQ,CKQT93|S:SJ963,H3...",4H,4.0,0.0,-450.0,4H@EW,0.0,1.0,0.0,0.0,0.0,15,25,-10,8,9,4.0,3.0,9.0,15,1
47504,5|J-turkey|J-norway,tistis_eyc98_A1a11212.pbn,open,W,0.0,9.0,"N:S9862,HT3,D92,CA7654|E:SAJ5,HQ5,D87643,CQJT|S:ST74,HKJ...",3N,3.0,0.0,-400.0,3N@EW,0.0,0.0,0.0,0.0,1.0,11,29,-18,8,10,4.0,5.0,9.0,11,2


=== Tidak cocok par (0): 5 contoh ===


,_board_number,_source_file,_room,_declarer,_result,_tricks_made,_deal_key,target_base,dd_par_level,dd_par_declarer_is_ns,dd_par_score,dd_par_contracts,dd_par_denom_S,dd_par_denom_H,dd_par_denom_D,dd_par_denom_C,dd_par_denom_N,ns_hcp,ew_hcp,ns_hcp_advantage,ns_best_fit,ew_best_fit,ns_dd_N,ns_dd_S,ew_dd_N,auction_len,opening_level
37142,12|Eddy Manoppo|Henky Lasut|Kazuo Furuta|Kenji Miyakuni,angelfire_nec2001_09INDSLA.PBN,open,E,0.0,9.0,"N:SQ7,HAK54,DK8532,C96|E:SKJ32,HQT3,DQT96,CAJ|S:S965,HJ9...",3S,1.0,0.0,-140.0,1S@EW,1.0,0.0,0.0,0.0,0.0,19,21,-2,8,8,5.0,4.0,7.0,10,1
33251,18|Bates|Katz|Silverman|Wolfson,angelfire_itt99_99ITTF02.PBN,open,N,-2.0,5.0,"N:SQ83,HQ2,DKQ9873,CQ3|E:SJ974,H,DA64,CJ87642|S:SAKT,HKJ...",1N,3.0,0.0,-130.0,3C@EW,0.0,0.0,0.0,1.0,0.0,22,18,4,9,9,5.0,5.0,8.0,7,1
887,24,85248.lin,closed,W,0.0,8.0,"N:SKQT3,HA2,DJ6542,C52|E:SJ7,HJT983,DA83,CKJ3|S:S82,HKQ7...",2H,4.0,0.0,-130.0,4C@EW,0.0,0.0,0.0,1.0,0.0,17,23,-6,9,8,5.0,4.0,8.0,8,1
47665,12|J-turkey|J-italy,tistis_eyc98_A1a11919.pbn,open,N,0.0,7.0,"N:SA53,HK64,DAQT3,CQ62|E:SQ42,HQ75,DJ7,CAKT54|S:SJ97,HJ3...",1N,1.0,0.0,-140.0,1H@EW|1S@EW,1.0,0.0,0.0,0.0,0.0,20,20,0,9,8,6.0,4.0,6.0,5,1
23890,19|Gal P|Kirchhoff E|Paulissen G|Winkler G,angelfire_etc01_04HUNNET.PBN,open,E,-2.0,6.0,"N:SK87,HKJ72,D63,CJ764|E:SJ96,HA983,DQT87,C93|S:SAT3,H65...",2H,3.0,0.0,-110.0,3D@EW,0.0,0.0,1.0,0.0,0.0,18,22,-4,9,8,5.0,5.0,6.0,11,1


---
## 10. Contoh Satu Papan Fisik: Open vs Closed Room

Di BBO vugraph setiap papan dimainkan dua kali (open & closed room) dengan
kartu identik — itulah alasan split dibuat *group-aware*. Tabel di bawah
menunjukkan kedua baris untuk beberapa papan: fitur tangan identik, yang
berbeda adalah lelang/kontrak yang dibid (dan karena itu labelnya).

In [22]:
grp_key = df['_source_file'].astype(str) + '||' + df['_board_number'].astype(str)
pair_keys = grp_key[grp_key.map(grp_key.value_counts()) == 2]
diff_label = df.loc[pair_keys.index].groupby(pair_keys)['matches_par_contract'].nunique()
example_keys = diff_label[diff_label == 2].index[:3]

cmp_cols = (['_source_file', '_board_number', '_room', 'target_base', 'matches_par_contract',
             'ns_hcp', 'ew_hcp', 'ns_dd_N', 'auction_len', 'auction_competitive', 'opening_level'])
cmp_cols = [c for c in cmp_cols if c in df.columns]
print(f'Papan dengan tepat 2 baris (open+closed): {pair_keys.nunique():,}')
print(f'...yang labelnya berbeda antar room     : {(diff_label == 2).sum():,}')
display(df.loc[grp_key.isin(example_keys), cmp_cols].sort_values(['_source_file', '_board_number', '_room']))

Papan dengan tepat 2 baris (open+closed): 9,070
...yang labelnya berbeda antar room     : 1,825


,_source_file,_board_number,_room,target_base,matches_par_contract,ns_hcp,ew_hcp,ns_dd_N,auction_len,auction_competitive,opening_level
69,85171.lin,6,closed,2S,0,23,17,9.0,8,0,1
68,85171.lin,6,open,3N,1,23,17,9.0,11,1,2
92,85173.lin,7,closed,3C,1,21,19,5.0,10,0,1
91,85173.lin,7,open,2S,0,21,19,5.0,8,0,1
96,85173.lin,9,closed,3D,1,20,20,8.0,12,1,1
95,85173.lin,9,open,2H,0,20,20,8.0,11,0,1


---
## Ringkasan

- Tabel mentah (bagian 1) menunjukkan data asli sebelum diolah: tangan, lelang, kontrak, hasil.
- Dataset final: lihat bagian 2 untuk ukuran split dan bagian 4 untuk kamus data lengkap
  (tersimpan di `outputs/results/nb01b_data_dictionary.csv`).
- 174 fitur model bebas missing value dan seluruhnya numerik (bagian 6); missing hanya
  ada di kolom metadata (`_room`, `_result`, `_tricks_made`, `_declarer`), yang tidak dipakai model.
- Visualisasi & analisis fitur lebih lanjut ada di `02_eda_features.ipynb`.